In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import optuna

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
 
IMG_SIZE = 96

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
train_data = pd.read_csv(r"C:\Learning_ML\DL_projects\Fashion_MNIST\fashionmnist\versions\4\fashion-mnist_train.csv")
test_data = pd.read_csv(r"C:\Learning_ML\DL_projects\Fashion_MNIST\fashionmnist\versions\4\fashion-mnist_test.csv")

In [4]:
xtrain = train_data.iloc[:, 1:].to_numpy(dtype=np.uint8)
ytrain = train_data.iloc[:, 0].to_numpy(dtype=np.int64)
xtest = test_data.iloc[:, 1:].to_numpy(dtype=np.uint8)
ytest = test_data.iloc[:, 0].to_numpy(dtype=np.int64)

In [5]:
x_train, x_val, y_train, y_val = train_test_split(xtrain, ytrain, test_size=0.1667, stratify=ytrain)

In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),          # safe for clothing, not for digits
    transforms.RandomRotation(10),
    transforms.Grayscale(num_output_channels=3), # pretrained nets expect 3 channels
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
 
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [8]:
class FashionMNISTImages(Dataset):
    def __init__(self, features, labels, transform):
        self.features = features.reshape(-1, 28, 28)  # uint8, HxW
        self.labels = labels
        self.transform = transform
 
    def __len__(self):
        return len(self.features)
 
    def __getitem__(self, idx):
        img = self.features[idx]                # (28, 28) uint8
        img = self.transform(img)                # -> (3, 96, 96) normalized tensor
        return img, torch.tensor(self.labels[idx], dtype=torch.long)
 

In [9]:
training = FashionMNISTImages(x_train, y_train, train_transform)
validation = FashionMNISTImages(x_val, y_val, eval_transform)
testing = FashionMNISTImages(xtest, ytest, eval_transform)

In [10]:
def build_model(name, num_classes, unfreeze_last_n, dropout):
    if name == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        blocks = [model.layer1, model.layer2, model.layer3, model.layer4]
 
    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        blocks = list(model.features.children())
 
    else:
        raise ValueError(f"Unknown model: {name}")
 
    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False
    # Unfreeze classifier head
    head = model.fc if name == "resnet18" else model.classifier
    for p in head.parameters():
        p.requires_grad = True
    # Unfreeze the last N backbone blocks for fine-tuning
    for block in blocks[-unfreeze_last_n:]:
        for p in block.parameters():
            p.requires_grad = True
 
    return model.to(device)

In [11]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, scheduler=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
 
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
 
        with torch.set_grad_enabled(train_mode):
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                out = model(xb)
                loss = criterion(out, yb)
 
            if train_mode:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()
 
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
 
    return total_loss / total, correct / total

In [ ]:
def objective(trial):
    model_name = trial.suggest_categorical("model_name", ["resnet18", "efficientnet_b0"])
    unfreeze_last_n = trial.suggest_int("unfreeze_last_n", 1, 2)  # keep most of backbone frozen
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])  # smaller than digit-CNN due to 96x96x3 images
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
 
    train_load = DataLoader(training, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_load = DataLoader(validation, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
 
    model = build_model(model_name, num_classes=10, unfreeze_last_n=unfreeze_last_n, dropout=dropout)
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
 
    n_epochs = 4  # fine-tuning converges fast since most weights are pretrained
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_load), epochs=n_epochs
    )
 
    for epoch in range(n_epochs):
        run_epoch(model, train_load, criterion, optimizer, scaler, scheduler)
        _, val_acc = run_epoch(model, val_load, criterion)
 
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
 
    return val_acc

In [13]:
if __name__ == "__main__":
    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=12)  # fewer trials needed — search space is smaller, trials are pricier
 
    print("Best trial:", study.best_trial.params)
    print("Best val accuracy:", study.best_value)
 
    p = study.best_trial.params
    train_load = DataLoader(training, batch_size=p["batch_size"], shuffle=True, num_workers=0, pin_memory=True)
    val_load = DataLoader(validation, batch_size=p["batch_size"], shuffle=False, num_workers=0, pin_memory=True)
    test_load = DataLoader(testing, batch_size=p["batch_size"], shuffle=False, num_workers=0, pin_memory=True)
 
    model = build_model(p["model_name"], num_classes=10, unfreeze_last_n=p["unfreeze_last_n"], dropout=p["dropout"])
    criterion = nn.CrossEntropyLoss()
    trainable_params = [pp for pp in model.parameters() if pp.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=p["lr"], weight_decay=p["weight_decay"])
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
 
    final_epochs = 12
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=p["lr"], steps_per_epoch=len(train_load), epochs=final_epochs)
 
    for epoch in range(final_epochs):
        train_loss, train_acc = run_epoch(model, train_load, criterion, optimizer, scaler, scheduler)
        val_loss, val_acc = run_epoch(model, val_load, criterion)
        print(f"Epoch {epoch+1}/{final_epochs}: "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
 
    test_loss, test_acc = run_epoch(model, test_load, criterion)
    print(f"\nFinal test accuracy: {test_acc:.4f}")

[I 2026-08-17 00:29:52,599] A new study created in memory with name: no-name-4c98c511-78ca-4961-8332-03e6f361de16
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\vivek/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:09<00:00, 2.20MB/s]
C:\Users\vivek\AppData\Local\Temp\ipykernel_29020\962316611.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
[I 2026-08-17 00:34:10,902] Trial 0 finished with value: 0.861127774445111 and parameters: {'model_name': 'efficientnet_b0', 'unfreeze_last_n': 1, 'lr': 0.000758740445770147, 'dropout': 0.5, 'batch_size': 64, 'weight_decay': 5.263292887332637e-06}. Best is trial 0 with value: 0.861127774445111.
[I 2026-08-17 00:38:08,282] Trial 1 finished with value: 0.881123775244951 and parameters: {'model_name': 'effici

Best trial: {'model_name': 'resnet18', 'unfreeze_last_n': 2, 'lr': 0.0010349477625827992, 'dropout': 0.1, 'batch_size': 128, 'weight_decay': 7.867342235038732e-06}
Best val accuracy: 0.9453109378124375


C:\Users\vivek\AppData\Local\Temp\ipykernel_29020\1554354982.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


Epoch 1/12: train_loss=0.4562 train_acc=0.8415 val_loss=0.2950 val_acc=0.8971
Epoch 2/12: train_loss=0.2677 train_acc=0.9035 val_loss=0.2289 val_acc=0.9144
Epoch 3/12: train_loss=0.2406 train_acc=0.9124 val_loss=0.2442 val_acc=0.9096
Epoch 4/12: train_loss=0.2168 train_acc=0.9234 val_loss=0.2058 val_acc=0.9278
Epoch 5/12: train_loss=0.1859 train_acc=0.9331 val_loss=0.1819 val_acc=0.9352
Epoch 6/12: train_loss=0.1667 train_acc=0.9398 val_loss=0.1773 val_acc=0.9345
Epoch 7/12: train_loss=0.1427 train_acc=0.9486 val_loss=0.1666 val_acc=0.9430
Epoch 8/12: train_loss=0.1192 train_acc=0.9565 val_loss=0.1594 val_acc=0.9446
Epoch 9/12: train_loss=0.0977 train_acc=0.9639 val_loss=0.1738 val_acc=0.9440
Epoch 10/12: train_loss=0.0728 train_acc=0.9738 val_loss=0.1649 val_acc=0.9495
Epoch 11/12: train_loss=0.0543 train_acc=0.9801 val_loss=0.1673 val_acc=0.9500
Epoch 12/12: train_loss=0.0436 train_acc=0.9849 val_loss=0.1681 val_acc=0.9512

Final test accuracy: 0.9499


In [14]:
study.best_params

{'model_name': 'resnet18',
 'unfreeze_last_n': 2,
 'lr': 0.0010349477625827992,
 'dropout': 0.1,
 'batch_size': 128,
 'weight_decay': 7.867342235038732e-06}

In [15]:
study.best_value

0.9453109378124375